# External Data Integration for IPL Dataset

Fetch and integrate:
1. **Player Metadata**: Age, nationality, batting/bowling style, role from ESPN Cricinfo
2. **Weather Data**: Temperature, humidity, conditions from Open-Meteo (free historical weather API)

Per hackathon rules: External data allowed if integrated programmatically.

In [1]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import time
from datetime import datetime
import re
from tqdm import tqdm
import json

## Load Base Dataset

In [2]:
df = pd.read_csv('../aggregate_player_match_features.csv')
df['date'] = pd.to_datetime(df['date'])
print(f"Base dataset: {df.shape}")
print(f"Unique players: {df['player'].nunique()}")
print(f"Unique matches: {df['match_id'].nunique()}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")

Base dataset: (6813, 34)
Unique players: 512
Unique matches: 291
Date range: 2022-03-26 00:00:00 to 2025-06-03 00:00:00


## 1. Weather Data from Open-Meteo API

Open-Meteo provides free historical weather data without API key.

In [3]:
# IPL venue coordinates (latitude, longitude)
venue_coordinates = {
    'Wankhede Stadium, Mumbai': (19.0083, 72.8331),
    'Brabourne Stadium, Mumbai': (18.9398, 72.8253),
    'Dr DY Patil Sports Academy, Mumbai': (19.0380, 73.0260),
    'Maharashtra Cricket Association Stadium, Pune': (18.5593, 73.9114),
    'Eden Gardens, Kolkata': (22.5645, 88.3433),
    'Narendra Modi Stadium, Ahmedabad': (23.0828, 72.5890),
    'Punjab Cricket Association IS Bindra Stadium, Mohali': (30.6900, 76.7370),
    'Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow': (26.9020, 80.9730),
    'Rajiv Gandhi International Stadium, Hyderabad': (17.4064, 78.5515),
    'M.Chinnaswamy Stadium, Bengaluru': (12.9789, 77.5996),
    'MA Chidambaram Stadium, Chennai': (13.0627, 80.2795),
    'Arun Jaitley Stadium, Delhi': (28.6375, 77.2426),
    'Barsapara Cricket Stadium, Guwahati': (26.1332, 91.7832),
    'Sawai Mansingh Stadium, Jaipur': (26.8933, 75.8094),
    'Himachal Pradesh Cricket Association Stadium, Dharamsala': (32.1956, 76.3257),
    'Maharaja Yadavindra Singh International Cricket Stadium, Mullanpur, Chandigarh': (30.7570, 76.7240),
    'Maharaja Yadavindra Singh International Cricket Stadium, Mullanpur, New Chandigarh': (30.7570, 76.7240),
    'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam': (17.7462, 83.3048)
}

print(f"Mapped {len(venue_coordinates)} venues")

Mapped 18 venues


In [4]:
def fetch_weather_for_match(date, venue, venue_coords):
    """
    Fetch historical weather from Open-Meteo API.
    
    Returns: dict with temperature, humidity, precipitation, weather_code
    """
    if venue not in venue_coords:
        return None
    
    lat, lon = venue_coords[venue]
    date_str = date.strftime('%Y-%m-%d')
    
    url = f"https://archive-api.open-meteo.com/v1/archive"
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': date_str,
        'end_date': date_str,
        'daily': 'temperature_2m_max,temperature_2m_min,precipitation_sum,weathercode',
        'timezone': 'Asia/Kolkata'
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if 'daily' in data:
                daily = data['daily']
                return {
                    'temp_max': daily['temperature_2m_max'][0] if daily['temperature_2m_max'] else None,
                    'temp_min': daily['temperature_2m_min'][0] if daily['temperature_2m_min'] else None,
                    'precipitation': daily['precipitation_sum'][0] if daily['precipitation_sum'] else None,
                    'weather_code': daily['weathercode'][0] if daily['weathercode'] else None
                }
    except Exception as e:
        print(f"Error fetching weather for {date_str} at {venue}: {e}")
    
    return None

# Test with one match
test_date = df['date'].iloc[0]
test_venue = df['venue'].iloc[0]
test_weather = fetch_weather_for_match(test_date, test_venue, venue_coordinates)
print(f"Test weather fetch for {test_venue} on {test_date.date()}:")
print(test_weather)

Test weather fetch for Wankhede Stadium, Mumbai on 2022-03-26:
{'temp_max': 31.3, 'temp_min': 23.9, 'precipitation': 0.0, 'weather_code': 3}


In [5]:
# Fetch weather for all unique match dates/venues
unique_matches = df[['match_id', 'date', 'venue']].drop_duplicates('match_id')
print(f"Fetching weather for {len(unique_matches)} matches...")

weather_cache = {}

for idx, row in tqdm(unique_matches.iterrows(), total=len(unique_matches)):
    key = (row['date'].date(), row['venue'])
    if key not in weather_cache:
        weather = fetch_weather_for_match(row['date'], row['venue'], venue_coordinates)
        weather_cache[key] = weather
        time.sleep(0.5)  # Rate limiting

print(f"\nFetched weather for {len(weather_cache)} unique date-venue pairs")
print(f"Success rate: {sum(1 for v in weather_cache.values() if v is not None) / len(weather_cache) * 100:.1f}%")

Fetching weather for 291 matches...


100%|██████████| 291/291 [06:53<00:00,  1.42s/it]


Fetched weather for 291 unique date-venue pairs
Success rate: 100.0%


In [9]:
# Add weather columns to dataframe
df['temp_max'] = df.apply(lambda row: weather_cache.get((row['date'].date(), row['venue']), {}).get('temp_max') if weather_cache.get((row['date'].date(), row['venue'])) else None, axis=1)
df['temp_min'] = df.apply(lambda row: weather_cache.get((row['date'].date(), row['venue']), {}).get('temp_min') if weather_cache.get((row['date'].date(), row['venue'])) else None, axis=1)
df['precipitation'] = df.apply(lambda row: weather_cache.get((row['date'].date(), row['venue']), {}).get('precipitation') if weather_cache.get((row['date'].date(), row['venue'])) else None, axis=1)
df['weather_code'] = df.apply(lambda row: weather_cache.get((row['date'].date(), row['venue']), {}).get('weather_code') if weather_cache.get((row['date'].date(), row['venue'])) else None, axis=1)

print(f"Weather columns added:")
print(f"  temp_max: {df['temp_max'].notna().sum()} non-null ({df['temp_max'].notna().mean()*100:.1f}%)")
print(f"  temp_min: {df['temp_min'].notna().sum()} non-null ({df['temp_min'].notna().mean()*100:.1f}%)")
print(f"  precipitation: {df['precipitation'].notna().sum()} non-null ({df['precipitation'].notna().mean()*100:.1f}%)")
print(f"  weather_code: {df['weather_code'].notna().sum()} non-null ({df['weather_code'].notna().mean()*100:.1f}%)")

Weather columns added:
  temp_max: 6813 non-null (100.0%)
  temp_min: 6813 non-null (100.0%)
  precipitation: 6813 non-null (100.0%)
  weather_code: 6813 non-null (100.0%)


## 2. Player Metadata from Cricbuzz / Fallback

Scraping player data is complex due to:
- Name matching ("Gaikwad" vs "Ruturaj Gaikwad")
- Rate limiting
- Site structure changes

We'll create a simpler approach: map common IPL players manually using known rosters.

In [ ]:
# Scrape player metadata from Wikipedia
# Wikipedia is more stable and has structured infoboxes for most cricket players

def scrape_player_from_wikipedia(player_name):
    """
    Scrape player metadata from Wikipedia.
    
    Returns: dict with role, batting_style, bowling_style, nationality, birth_date
    """
    # Wikipedia URL - try with underscores
    wiki_url = f"https://en.wikipedia.org/wiki/{player_name.replace(' ', '_')}"
    
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        response = requests.get(wiki_url, headers=headers, timeout=10)
        
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Find infobox (cricket player info)
            infobox = soup.find('table', class_='infobox')
            if not infobox:
                return None
            
            metadata = {
                'role': None,
                'batting_style': None,
                'bowling_style': None,
                'nationality': None,
                'birth_date': None
            }
            
            # Extract from infobox rows
            rows = infobox.find_all('tr')
            for row in rows:
                th = row.find('th')
                td = row.find('td')
                
                if th and td:
                    label = th.get_text(strip=True).lower()
                    value = td.get_text(strip=True)
                    
                    # Extract role
                    if 'role' in label or 'playing role' in label:
                        metadata['role'] = value
                    
                    # Extract batting style
                    elif 'batting' in label and 'average' not in label:
                        metadata['batting_style'] = value
                    
                    # Extract bowling style
                    elif 'bowling' in label and 'average' not in label:
                        # Clean up bowling style
                        bowling = value.replace('Right-arm', 'Right-arm ').replace('Left-arm', 'Left-arm ')
                        metadata['bowling_style'] = bowling
                    
                    # Extract birth date and nationality
                    elif 'born' in label:
                        # Try to parse birth date
                        # Common formats: "5 November 1988", "(1988-11-05) 5 November 1988"
                        import re
                        from datetime import datetime
                        
                        # Look for date in parentheses first (more reliable)
                        paren_match = re.search(r'\((\d{4})-(\d{2})-(\d{2})\)', value)
                        if paren_match:
                            year, month, day = paren_match.groups()
                            try:
                                metadata['birth_date'] = f"{year}-{month}-{day}"
                            except:
                                pass
                        else:
                            # Try to parse text date
                            date_patterns = [
                                r'(\d{1,2})\s+(January|February|March|April|May|June|July|August|September|October|November|December)\s+(\d{4})',
                                r'(January|February|March|April|May|June|July|August|September|October|November|December)\s+(\d{1,2}),?\s+(\d{4})'
                            ]
                            for pattern in date_patterns:
                                match = re.search(pattern, value)
                                if match:
                                    try:
                                        date_str = match.group(0)
                                        parsed_date = pd.to_datetime(date_str)
                                        metadata['birth_date'] = parsed_date.strftime('%Y-%m-%d')
                                        break
                                    except:
                                        pass
                        
                        # Extract nationality from birth location
                        if 'India' in value:
                            metadata['nationality'] = 'India'
                        elif 'Australia' in value:
                            metadata['nationality'] = 'Australia'
                        elif 'England' in value:
                            metadata['nationality'] = 'England'
                        elif 'South Africa' in value:
                            metadata['nationality'] = 'South Africa'
                        elif 'New Zealand' in value:
                            metadata['nationality'] = 'New Zealand'
                        elif 'West Indies' in value:
                            metadata['nationality'] = 'West Indies'
                        elif 'Pakistan' in value:
                            metadata['nationality'] = 'Pakistan'
                        elif 'Sri Lanka' in value:
                            metadata['nationality'] = 'Sri Lanka'
                        elif 'Afghanistan' in value:
                            metadata['nationality'] = 'Afghanistan'
                        elif 'Bangladesh' in value:
                            metadata['nationality'] = 'Bangladesh'
            
            return metadata if any(metadata.values()) else None
    
    except Exception as e:
        # Silent fail - many players won't have Wikipedia pages
        pass
    
    return None


# Test with one player
test_player = "Virat Kohli"
print(f"Testing scraper with: {test_player}")
test_metadata = scrape_player_from_wikipedia(test_player)
print(f"Result: {test_metadata}")

# Test with a player from the dataset
test_player2 = df['player'].iloc[0]
print(f"\nTesting with dataset player: {test_player2}")
test_metadata2 = scrape_player_from_wikipedia(test_player2)
print(f"Result: {test_metadata2}")

In [19]:
# Batch scrape all unique players from Wikipedia
unique_players = df['player'].unique()
print(f"Scraping metadata for {len(unique_players)} unique players from Wikipedia...")
print("This will take ~15-20 minutes with rate limiting...")

player_metadata_cache = {}

for i, player in enumerate(tqdm(unique_players), 1):
    if player not in player_metadata_cache:
        metadata = scrape_player_from_wikipedia(player)
        player_metadata_cache[player] = metadata
        
        # Rate limiting - be respectful to Wikipedia servers
        time.sleep(1.5)  # 1.5 seconds between requests
        
        # Progress checkpoint every 50 players
        if i % 50 == 0:
            successful = sum(1 for v in player_metadata_cache.values() if v is not None)
            print(f"\nCheckpoint: {i}/{len(unique_players)} players scraped")
            print(f"Success rate so far: {successful / len(player_metadata_cache) * 100:.1f}%")
            print(f"Successful: {successful}, Failed: {len(player_metadata_cache) - successful}")

print(f"\n{'='*70}")
print(f"Scraping complete!")
print(f"Total players: {len(player_metadata_cache)}")
successful_scrapes = sum(1 for v in player_metadata_cache.values() if v is not None)
print(f"Successful scrapes: {successful_scrapes} ({successful_scrapes/len(player_metadata_cache)*100:.1f}%)")
print(f"Failed scrapes: {len(player_metadata_cache) - successful_scrapes}")

# Save cache to avoid re-scraping
with open('../../player_metadata_cache.json', 'w') as f:
    json.dump(player_metadata_cache, f, indent=2)
print(f"\nCache saved to: player_metadata_cache.json")

# Show sample of successful scrapes
print(f"\nSample successful scrapes:")
success_count = 0
for player, metadata in player_metadata_cache.items():
    if metadata and success_count < 5:
        print(f"  {player}: {metadata}")
        success_count += 1

Scraping metadata for 512 unique players from Wikipedia...
This will take ~15-20 minutes with rate limiting...


 10%|▉         | 50/512 [02:34<21:01,  2.73s/it]


Checkpoint: 50/512 players scraped
Success rate so far: 52.0%
Successful: 26, Failed: 24


 20%|█▉        | 100/512 [05:04<19:40,  2.87s/it]


Checkpoint: 100/512 players scraped
Success rate so far: 45.0%
Successful: 45, Failed: 55


...


Checkpoint: 450/512 players scraped
Success rate so far: 35.8%
Successful: 161, Failed: 289


 98%|█████████▊| 500/512 [24:47<00:34,  2.92s/it]


Checkpoint: 500/512 players scraped
Success rate so far: 35.6%
Successful: 178, Failed: 322


100%|██████████| 512/512 [25:22<00:00,  2.97s/it]


Scraping complete!
Total players: 512
Successful scrapes: 182 (35.5%)
Failed scrapes: 330

Cache saved to: player_metadata_cache.json

Sample successful scrapes:
  Dhoni: {'role': 'Wicket-keeper-batter', 'batting_style': 'Right-handed', 'bowling_style': '–', 'nationality': 'India'}
  Dwayne Bravo: {'role': 'All-rounder', 'batting_style': 'Right-handed', 'bowling_style': '6/55', 'nationality': None}
  Nitish Rana: {'role': 'Batting allrounder', 'batting_style': 'Left-handed', 'bowling_style': 'Right-arm off break', 'nationality': 'India'}
  Shivam Dube: {'role': 'All-rounder', 'batting_style': 'Left-handed', 'bowling_style': '1/19', 'nationality': 'India'}
  Shivam Mavi: {'role': 'Bowler', 'batting_style': 'Right-handed', 'bowling_style': '4/22', 'nationality': 'India'}


In [ ]:
# Merge player metadata into dataframe and calculate age features
df['player_role'] = df['player'].map(lambda p: player_metadata_cache.get(p, {}).get('role') if player_metadata_cache.get(p) else None)
df['batting_style'] = df['player'].map(lambda p: player_metadata_cache.get(p, {}).get('batting_style') if player_metadata_cache.get(p) else None)
df['bowling_style'] = df['player'].map(lambda p: player_metadata_cache.get(p, {}).get('bowling_style') if player_metadata_cache.get(p) else None)
df['nationality'] = df['player'].map(lambda p: player_metadata_cache.get(p, {}).get('nationality') if player_metadata_cache.get(p) else None)
df['birth_date'] = df['player'].map(lambda p: player_metadata_cache.get(p, {}).get('birth_date') if player_metadata_cache.get(p) else None)

# Convert birth_date to datetime
df['birth_date'] = pd.to_datetime(df['birth_date'], errors='coerce')

# Calculate age at match date
df['age'] = (df['date'] - df['birth_date']).dt.days / 365.25

# Age squared (captures non-linear peak performance)
df['age_squared'] = df['age'] ** 2

# Career stage categorical feature
def get_career_stage(age):
    if pd.isna(age):
        return None
    elif age < 25:
        return 'Emerging'
    elif age <= 32:
        return 'Prime'
    else:
        return 'Veteran'

df['career_stage'] = df['age'].apply(get_career_stage)

print(f"Player metadata merged:")
print(f"  player_role: {df['player_role'].notna().sum()} non-null ({df['player_role'].notna().mean()*100:.1f}%)")
print(f"  batting_style: {df['batting_style'].notna().sum()} non-null ({df['batting_style'].notna().mean()*100:.1f}%)")
print(f"  bowling_style: {df['bowling_style'].notna().sum()} non-null ({df['bowling_style'].notna().mean()*100:.1f}%)")
print(f"  nationality: {df['nationality'].notna().sum()} non-null ({df['nationality'].notna().mean()*100:.1f}%)")
print(f"  birth_date: {df['birth_date'].notna().sum()} non-null ({df['birth_date'].notna().mean()*100:.1f}%)")
print(f"  age: {df['age'].notna().sum()} non-null ({df['age'].notna().mean()*100:.1f}%)")

print(f"\nAge statistics:")
print(f"  Min age: {df['age'].min():.1f} years")
print(f"  Max age: {df['age'].max():.1f} years")
print(f"  Mean age: {df['age'].mean():.1f} years")
print(f"  Median age: {df['age'].median():.1f} years")

print(f"\nCareer stage distribution:")
print(df['career_stage'].value_counts())

print(f"\nPlayer role distribution:")
print(df['player_role'].value_counts())

print(f"\nTop nationalities:")
print(df['nationality'].value_counts().head(10))

## 2b. Player Roles from Match Stats

The scraped `player_role` is missing for ~2/3 of players (surname-only names don't match Wikipedia pages) and uses ~30 inconsistent spellings.
Instead, derive one role per player from their IPL stats across all matches, as four one-hot columns (exactly one is 1):

- **wicketkeeper**: any stumping, or the scraped role mentions keeper
- **allrounder**: averages ≥ 0.8 overs *and* ≥ 6 balls faced per match
- **bowler**: meets the overs threshold only
- **batter**: everyone else

Thresholds were calibrated against the scraped roles we do have. Computed over all seasons (a small, accepted leak).

In [ ]:
ROLE_COLS = ['is_batter', 'is_bowler', 'is_allrounder', 'is_wicketkeeper']
MIN_OVERS_PER_MATCH = 0.8        # bowls regularly
MIN_BALLS_FACED_PER_MATCH = 6    # bats regularly

roles = df.groupby('player').agg(
    matches=('match_id', 'count'),
    overs_per_match=('overs_bowled', 'mean'),
    balls_faced_per_match=('balls_faced', 'mean'),
    stumpings=('stumpings', 'sum'),
    scraped_role=('player_role', 'first'),
)

scraped_keeper = roles['scraped_role'].map(
    lambda r: isinstance(r, str) and 'keeper' in re.sub(r'\[\d+\]', '', r).lower()
)
keeper = (roles['stumpings'] > 0) | scraped_keeper
bowls = roles['overs_per_match'] >= MIN_OVERS_PER_MATCH
bats = roles['balls_faced_per_match'] >= MIN_BALLS_FACED_PER_MATCH

roles['role'] = 'batter'
roles.loc[bowls & ~bats, 'role'] = 'bowler'
roles.loc[bowls & bats, 'role'] = 'allrounder'
roles.loc[keeper, 'role'] = 'wicketkeeper'
for col in ROLE_COLS:
    roles[col] = (roles['role'] == col.removeprefix('is_')).astype(int)

roles = roles.drop(columns='scraped_role').reset_index()
roles.to_csv('../roles.csv', index=False)

# Replace the scraped role string with the one-hot columns
df = df.drop(columns='player_role').merge(roles[['player'] + ROLE_COLS], on='player', how='left')

print(roles['role'].value_counts())
print(f"
Rows with exactly one role: {(df[ROLE_COLS].sum(axis=1) == 1).mean()*100:.1f}%")

## 3. Save Augmented Dataset

In [ ]:
# Save augmented dataset
output_path = '../aggregate_player_match_features_with_external_data.csv'

# Drop birth_date (intermediate column, keep age instead)
df_to_save = df.drop(columns=['birth_date'])

df_to_save.to_csv(output_path, index=False)

print(f"\n{'='*70}")
print(f"AUGMENTED DATASET SAVED")
print(f"{'='*70}")
print(f"Path: {output_path}")
print(f"Shape: {df_to_save.shape}")
print(f"\nNew columns added:")
original_cols = pd.read_csv('../aggregate_player_match_features.csv').columns
new_cols = [c for c in df_to_save.columns if c not in original_cols]
for col in new_cols:
    print(f"  - {col}")

print(f"\nSample augmented row:")
sample = df_to_save.iloc[100]
sample_role = next(c.removeprefix('is_') for c in ROLE_COLS if sample[c] == 1)
print(f"  Player: {sample['player']} ({sample_role}, {sample['nationality']})")
print(f"  Date: {sample['date']}")
print(f"  Age: {sample['age']:.1f} years ({sample['career_stage']})")
print(f"  Venue: {sample['venue']}")
print(f"  Weather: {sample['temp_max']:.1f}°C max, {sample['precipitation']:.1f}mm rain")
print(f"  Batting: {sample['batting_style']}, Bowling: {sample['bowling_style']}")
print(f"  Fantasy Points: {sample['fantasy_points']}")

print(f"\n{'='*70}")
print(f"SUMMARY")
print(f"{'='*70}")
print(f"Total features: {df_to_save.shape[1]}")
print(f"Original features: {len(original_cols)}")
print(f"New features: {len(new_cols)}")
print(f"\nExternal data integration complete!")

In [29]:
df = pd.read_csv(output_path)


In [36]:
df.iloc[123]

match_id                                                      202206
season                                                          2022
date                                                      2022-03-30
venue                             Dr DY Patil Sports Academy, Mumbai
player                                                    Rutherford
runs                                                            28.0
balls_faced                                                     40.0
fours                                                            1.0
sixes                                                            1.0
dismissal_type                                                caught
was_dismissed                                                   True
strike_rate                                                     70.0
balls_bowled                                                     0.0
runs_conceded                                                    0.0
wickets                           